# Document Clustering with LLM Embeddings

## Assignment (g): Document Clustering using State-of-the-Art Embeddings

**Author:** Nitish  
**Date:** December 2024

---

## Table of Contents
1. Introduction
2. Dataset Preparation
3. Sentence Transformers Embeddings
4. Document Clustering
5. Visualization with UMAP
6. Topic Analysis
7. Evaluation Metrics
8. Conclusion

In [ ]:
!pip install sentence-transformers numpy pandas matplotlib seaborn scikit-learn umap-learn datasets -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Base libraries imported!")

In [ ]:
from sentence_transformers import SentenceTransformer
import umap
print("Sentence Transformers and UMAP imported!")

## 1. Create Document Dataset

In [ ]:
# Create diverse document dataset
documents = [
    # Technology (0)
    "Artificial intelligence is transforming how we interact with computers and machines.",
    "Machine learning algorithms can identify patterns in large datasets automatically.",
    "Deep neural networks have revolutionized computer vision and natural language processing.",
    "Cloud computing enables scalable and flexible IT infrastructure for businesses.",
    "Cybersecurity threats are becoming more sophisticated with advanced hacking techniques.",
    "Blockchain technology provides decentralized and secure transaction records.",
    "Quantum computing promises to solve complex problems exponentially faster.",
    "The Internet of Things connects billions of devices worldwide.",
    "5G networks enable faster mobile connectivity and lower latency.",
    "Software development practices have evolved with agile methodologies.",
    # Sports (1)
    "The football team won the championship after an exciting final match.",
    "Basketball players train intensively to improve their shooting accuracy.",
    "Tennis requires excellent hand-eye coordination and physical endurance.",
    "Swimming is one of the best full-body workout exercises available.",
    "The Olympic Games bring together athletes from around the world.",
    "Soccer is the most popular sport globally with billions of fans.",
    "Golf courses are designed to challenge players with various obstacles.",
    "Marathon runners need exceptional cardiovascular fitness and mental strength.",
    "Cricket matches can last for several days in test format.",
    "Boxing requires both physical strength and strategic thinking.",
    # Science (2)
    "Climate change is causing rising sea levels and extreme weather events.",
    "DNA sequencing has revolutionized our understanding of genetics and heredity.",
    "The discovery of gravitational waves confirmed Einstein's predictions.",
    "Vaccines have saved millions of lives by preventing infectious diseases.",
    "Space exploration continues to reveal mysteries of our solar system.",
    "Renewable energy sources like solar and wind are becoming more efficient.",
    "Stem cell research offers potential treatments for many diseases.",
    "The periodic table organizes elements by their atomic properties.",
    "Evolution explains the diversity of life through natural selection.",
    "Particle physics explores the fundamental building blocks of matter.",
    # Finance (3)
    "Stock markets fluctuate based on economic indicators and investor sentiment.",
    "Cryptocurrency trading has become increasingly popular among investors.",
    "Interest rates affect borrowing costs and economic growth.",
    "Diversification helps reduce investment portfolio risk.",
    "Central banks use monetary policy to control inflation.",
    "Real estate investment provides both rental income and appreciation.",
    "Retirement planning requires long-term financial strategy and discipline.",
    "Corporate bonds offer fixed income with varying risk levels.",
    "Hedge funds use complex strategies to generate returns.",
    "Financial regulations protect consumers and maintain market stability.",
    # Health (4)
    "Regular exercise improves cardiovascular health and mental wellbeing.",
    "A balanced diet includes proteins, carbohydrates, fats, and vitamins.",
    "Mental health awareness has increased significantly in recent years.",
    "Sleep quality affects cognitive function and overall health.",
    "Meditation and mindfulness can reduce stress and anxiety.",
    "Preventive healthcare focuses on early detection and disease prevention.",
    "Antibiotics should be used responsibly to prevent resistance.",
    "Telemedicine enables remote healthcare consultations and monitoring.",
    "Nutrition plays a crucial role in immune system function.",
    "Physical therapy helps patients recover from injuries and surgeries."
]

categories = ['Technology']*10 + ['Sports']*10 + ['Science']*10 + ['Finance']*10 + ['Health']*10
labels = [0]*10 + [1]*10 + [2]*10 + [3]*10 + [4]*10

df = pd.DataFrame({'document': documents, 'category': categories, 'label': labels})
print(f"Dataset: {len(df)} documents, {len(df['category'].unique())} categories")
print(df['category'].value_counts())

## 2. Generate Embeddings with Sentence Transformers

In [ ]:
# Load pre-trained model
model_name = 'all-MiniLM-L6-v2'  # Fast and effective
model = SentenceTransformer(model_name)
print(f"Model loaded: {model_name}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
# Generate embeddings
embeddings = model.encode(documents, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Compute similarity matrix
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(embeddings)

plt.figure(figsize=(12, 10))
sns.heatmap(similarity_matrix, cmap='coolwarm', xticklabels=False, yticklabels=False)
plt.title('Document Similarity Matrix (Cosine Similarity)')
# Add category boundaries
for i in range(1, 5):
    plt.axhline(y=i*10, color='black', linewidth=2)
    plt.axvline(x=i*10, color='black', linewidth=2)
plt.tight_layout(); plt.show()

## 3. Dimensionality Reduction with UMAP

In [ ]:
# UMAP for visualization
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.1)
embeddings_2d = reducer.fit_transform(embeddings)
print(f"UMAP embeddings: {embeddings_2d.shape}")

In [ ]:
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=labels, cmap='tab10', s=100, alpha=0.7)
plt.colorbar(scatter, label='Category')
for i, cat in enumerate(['Tech', 'Sports', 'Science', 'Finance', 'Health']):
    idx = labels.index(i)
    plt.annotate(cat, (embeddings_2d[idx, 0], embeddings_2d[idx, 1]), fontsize=12, fontweight='bold')
plt.title('Document Embeddings (UMAP Projection)', fontsize=14)
plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2')
plt.tight_layout(); plt.show()

## 4. K-Means Clustering

In [ ]:
# Find optimal K
inertias, silhouettes = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(embeddings)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(embeddings, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method'); axes[0].axvline(x=5, color='r', linestyle='--')
axes[1].plot(K_range, silhouettes, 'go-'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Score'); axes[1].axvline(x=5, color='r', linestyle='--')
plt.tight_layout(); plt.show()

In [ ]:
# Apply K-Means with K=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

print("K-Means Clustering Results:")
print(f"  Silhouette Score: {silhouette_score(embeddings, cluster_labels):.4f}")
print(f"  Calinski-Harabasz: {calinski_harabasz_score(embeddings, cluster_labels):.4f}")
print(f"  Davies-Bouldin: {davies_bouldin_score(embeddings, cluster_labels):.4f}")
print(f"  ARI: {adjusted_rand_score(labels, cluster_labels):.4f}")
print(f"  NMI: {normalized_mutual_info_score(labels, cluster_labels):.4f}")

In [ ]:
# Visualize clusters
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=labels, cmap='tab10', s=100, alpha=0.7)
axes[0].set_title('True Categories', fontsize=14)

axes[1].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=cluster_labels, cmap='tab10', s=100, alpha=0.7)
axes[1].set_title('K-Means Clusters', fontsize=14)

plt.tight_layout(); plt.show()

## 5. Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

linkage_matrix = linkage(embeddings, method='ward')
plt.figure(figsize=(16, 8))
dendrogram(linkage_matrix, labels=categories, leaf_rotation=90, leaf_font_size=8)
plt.title('Hierarchical Clustering Dendrogram', fontsize=14)
plt.xlabel('Document'); plt.ylabel('Distance')
plt.tight_layout(); plt.show()

In [ ]:
agg = AgglomerativeClustering(n_clusters=5, linkage='ward')
agg_labels = agg.fit_predict(embeddings)
print(f"Hierarchical Clustering:")
print(f"  ARI: {adjusted_rand_score(labels, agg_labels):.4f}")
print(f"  NMI: {normalized_mutual_info_score(labels, agg_labels):.4f}")

## 6. Cluster Analysis

In [ ]:
# Analyze cluster composition
df['cluster'] = cluster_labels
print("\nCluster Composition:")
for c in range(5):
    cluster_docs = df[df['cluster'] == c]
    cat_dist = cluster_docs['category'].value_counts()
    print(f"\nCluster {c} ({len(cluster_docs)} docs):")
    for cat, count in cat_dist.items():
        print(f"  {cat}: {count}")

In [ ]:
# Find representative documents per cluster
print("\nRepresentative Documents per Cluster:")
for c in range(5):
    cluster_mask = cluster_labels == c
    cluster_embeddings = embeddings[cluster_mask]
    centroid = kmeans.cluster_centers_[c]
    distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)
    closest_idx = np.argmin(distances)
    doc_idx = np.where(cluster_mask)[0][closest_idx]
    print(f"\nCluster {c}: {documents[doc_idx][:80]}...")

## 7. Compare Different Embedding Models

In [ ]:
# Test with different models
model_names = ['all-MiniLM-L6-v2', 'paraphrase-MiniLM-L6-v2']
results = []

for mname in model_names:
    try:
        m = SentenceTransformer(mname)
        emb = m.encode(documents)
        km = KMeans(n_clusters=5, random_state=42, n_init=10)
        pred = km.fit_predict(emb)
        results.append({
            'Model': mname,
            'Silhouette': silhouette_score(emb, pred),
            'ARI': adjusted_rand_score(labels, pred),
            'NMI': normalized_mutual_info_score(labels, pred)
        })
    except Exception as e:
        print(f"Error with {mname}: {e}")

results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df.to_string(index=False))

## 8. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

# Create mapping from clusters to categories
cm = confusion_matrix(labels, cluster_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Cluster {i}' for i in range(5)],
            yticklabels=['Tech', 'Sports', 'Science', 'Finance', 'Health'])
plt.xlabel('Predicted Cluster'); plt.ylabel('True Category')
plt.title('Confusion Matrix: True Categories vs Clusters')
plt.tight_layout(); plt.show()

## 9. Conclusion

### Key Findings:
- **Sentence Transformers** provide high-quality semantic embeddings
- **UMAP** effectively visualizes high-dimensional embeddings
- **K-Means** achieves good clustering with proper K selection
- **LLM embeddings** capture semantic similarity well

### Applications:
- Document organization
- Topic discovery
- Content recommendation
- Search result clustering

In [ ]:
print("="*60)
print("DOCUMENT CLUSTERING WITH LLM EMBEDDINGS - COMPLETE")
print("="*60)
print("\n✓ Created document dataset (5 categories)")
print("✓ Generated embeddings with Sentence Transformers")
print("✓ Visualized with UMAP")
print("✓ Applied K-Means and Hierarchical clustering")
print("✓ Analyzed cluster composition")
print("✓ Comprehensive evaluation metrics")